In [6]:
#Intall Java
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk

Hit:1 https://nvidia.github.io/libnvidia-container/stable/deb/amd64  InRelease
Hit:2 https://download.docker.com/linux/debian bullseye InRelease              
Hit:3 https://snapshot.debian.org/archive/debian/20260815T000000Z bullseye InRelease
Hit:4 https://snapshot.debian.org/archive/debian-security/20260815T000000Z bullseye-security InRelease
Hit:5 https://snapshot.debian.org/archive/debian/20260815T000000Z bullseye-updates InRelease
Hit:6 https://packages.cloud.google.com/apt gcsfuse-bullseye InRelease
Hit:7 https://packages.cloud.google.com/apt google-compute-engine-bullseye-stable InRelease
Hit:8 https://packages.cloud.google.com/apt cloud-sdk-bullseye InRelease
Hit:9 https://packages.cloud.google.com/apt google-fast-socket InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk is already the newest version (17.0.20+8-1~deb11u1).
0 upgraded, 0 newly installed, 0 to remove and 8 not upg

In [1]:
# Verify that Java is installed
import os
import shutil
import subprocess

print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("Java executable:", shutil.which("java"))

subprocess.run(["bash", "-c", "java -version"])

JAVA_HOME: None
Java executable: /usr/bin/java


openjdk version "17.0.20" 2026-07-21
OpenJDK Runtime Environment (build 17.0.20+8-1-deb11u1-Debian)
OpenJDK 64-Bit Server VM (build 17.0.20+8-1-deb11u1-Debian, mixed mode, sharing)


CompletedProcess(args=['bash', '-c', 'java -version'], returncode=0)

In [2]:
#Configure JAVA_HOME

java_home = subprocess.check_output(
    ["bash", "-c", "dirname $(dirname $(readlink -f $(which java)))"],
    text=True
).strip()

os.environ["JAVA_HOME"] = java_home

print("JAVA_HOME:", os.environ["JAVA_HOME"])

JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


In [1]:
#Start a spark session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataprocTest") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/15 17:14:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.4
Spark master: local[*]


In [3]:
#Create a sample dataset
data = [
    (1, "Alice", "Engineering", 95000),
    (2, "Bob", "Marketing", 75000),
    (3, "Charlie", "Engineering", 105000),
    (4, "David", "Finance", 85000),
    (5, "Emma", "Marketing", 80000),
    (6, "Frank", "Finance", 90000)
]

columns = ["id", "name", "department", "salary"]

df = spark.createDataFrame(data, columns)

df.show()

+---+-------+-----------+------+
| id|   name| department|salary|
+---+-------+-----------+------+
|  1|  Alice|Engineering| 95000|
|  2|    Bob|  Marketing| 75000|
|  3|Charlie|Engineering|105000|
|  4|  David|    Finance| 85000|
|  5|   Emma|  Marketing| 80000|
|  6|  Frank|    Finance| 90000|
+---+-------+-----------+------+



In [4]:
# Run a simple aggregation - calculate the average salary by department
from pyspark.sql.functions import avg, count

result = df.groupBy("department").agg(
    count("*").alias("employee_count"),
    avg("salary").alias("average_salary")
)

result.show()

[Stage 3:>                                                          (0 + 8) / 8]

+-----------+--------------+--------------+
| department|employee_count|average_salary|
+-----------+--------------+--------------+
|Engineering|             2|      100000.0|
|  Marketing|             2|       77500.0|
|    Finance|             2|       87500.0|
+-----------+--------------+--------------+



In [5]:
#Verify Spark execution
print("Total employees:", df.count())
print("Spark application ID:", spark.sparkContext.applicationId)

Total employees: 6
Spark application ID: local-1794682135728


In [6]:
#Test sparksql
df.createOrReplaceTempView("employees")

spark.sql("""
    SELECT department,
           COUNT(*) AS employees,
           AVG(salary) AS avg_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
""").show()

+-----------+---------+----------+
| department|employees|avg_salary|
+-----------+---------+----------+
|Engineering|        2|  100000.0|
|    Finance|        2|   87500.0|
|  Marketing|        2|   77500.0|
+-----------+---------+----------+



In [10]:
#Download the CSV from GCS - this is a workaround, not best practice; do not do this with large files
from google.cloud import storage

client = storage.Client()

bucket = client.bucket("adsp-31011-ip07-f26-bucket")
blob = bucket.blob("employees_sample.csv")

blob.download_to_filename("/tmp/employees_sample.csv")

In [12]:
#Load the file into a Spark DataFrame
df_csv = spark.read.csv(
    "file:///tmp/employees_sample.csv",
    header=True,
    inferSchema=True
)

df_csv.show()

+---+-------+-----------+------+
| id|   name| department|salary|
+---+-------+-----------+------+
|  1|  Alice|Engineering| 95000|
|  2|    Bob|  Marketing| 75000|
|  3|Charlie|Engineering|105000|
|  4|  David|    Finance| 85000|
|  5|   Emma|  Marketing| 80000|
|  6|  Frank|    Finance| 90000|
+---+-------+-----------+------+



In [14]:
#Run a simple Spark operation
df_csv.groupBy("department").avg("salary").show()

+-----------+-----------+
| department|avg(salary)|
+-----------+-----------+
|Engineering|   100000.0|
|    Finance|    87500.0|
|  Marketing|    77500.0|
+-----------+-----------+

